In [ ]:
import pandas as pd
import numpy as np
from google.cloud import storage
import tempfile
import json
import os
import importlib
import SignalProcessing_022726
importlib.reload(SignalProcessing_022726)
from SignalProcessing_022726 import run_signal_processing

LOCAL_CSV_PATH = r"C:\Users\ahasa\Downloads\001_BicepCurl_L_T.csv"   # <-- CHANGE THIS
BUCKET_NAME = "rehab-imu-data-arshaan-2026"   # <-- CHANGE THIS

def load_csv(file_path):
    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna()
    return df

def process_imu_file(file_path):
    df = load_csv(file_path)

    # ---- CALL YOUR FULL MODEL ----
    results = run_signal_processing(df)

    return results

def upload_file_to_gcs(local_file, bucket_name, destination_blob):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob)
    blob.upload_from_filename(local_file)
    print(f"Uploaded {local_file} to {destination_blob}")

def run_pipeline():
    print("Processing file...")

    results = process_imu_file(LOCAL_CSV_PATH)
    print("Results:", results)

    upload_file_to_gcs(
        LOCAL_CSV_PATH,
        BUCKET_NAME,
        f"raw/{os.path.basename(LOCAL_CSV_PATH)}"
    )

    with tempfile.NamedTemporaryFile(delete=False, suffix=".json") as tmp:
        with open(tmp.name, "w") as f:
            json.dump(results, f)
        temp_json_path = tmp.name

    upload_file_to_gcs(
        temp_json_path,
        BUCKET_NAME,
        f"processed/{os.path.basename(LOCAL_CSV_PATH)}.json"
    )

    print("Pipeline complete.")

run_pipeline()

In [ ]:
import pandas as pd
import importlib
import SignalProcessing_022726

importlib.reload(SignalProcessing_022726)

from SignalProcessing_022726 import run_signal_processing

df = pd.read_csv(r"C:\Users\ahasa\Downloads\001_BicepCurl_L_T.csv")
print(run_signal_processing(df))

In [ ]:
import SignalProcessing_022726

print(dir(SignalProcessing_022726))

In [ ]:
import SignalProcessing_022726
print(SignalProcessing_022726.__file__)
print(dir(SignalProcessing_022726))

In [ ]:
!pip install google-cloud-storage

In [ ]:
!pip install "protobuf>=3.20.3,<5.0.0" --force-reinstall

In [ ]:
!pip install google-cloud-storage --upgrade --no-deps